In [3]:
import pandas as pd 
import sklearn as sk
import os
import pandas as pd
import numpy as np 
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
import scanpy as sc
import anndata as ad
import bbknn
import pandas as pd
import os

In [20]:
ensg_trad='/mnt/cold1/snaketree/prj/scRNA/dataset/rCASC_GSE144735/first_ensg_gs_map.tsv'
ens=pd.read_csv(ensg_trad,sep='\t')
mapping_dict = dict(zip(ens['ens'], ens['gs']))

In [38]:
#leggo dato intero e faccio check su geni variabili
samples=['CRC0322_NT_1_3000','CRC0327_NT_2','CRC0542_NT72h_1']
DIC_FOLDER={
 'CRC0322_NT_1_3000':'rCASC_Ire_cetuxi',
 'CRC0327_NT_2':'rCASC_Ire_cetuxi',
 'CRC0542_NT72h_1':'rCASC_longer'
 }
PRJ_ROOT='/mnt/cold1/snaketree/prj/scRNA'
dic_path={}
data=pd.DataFrame()
i=0
for sample in samples:
    dic_path[sample]=PRJ_ROOT+'/dataset/'+DIC_FOLDER[sample]+'/'+sample+'_dir/annotated_'+sample+'_log2_pc1_cpm.csv'

for sample_name, file_path in dic_path.items():
    print(f"Processing {sample_name}...")
    df = pd.read_csv(file_path,header=0,index_col=0,sep=',').T
    df = df[[col for col in df.columns if col.split(":")[0] in mapping_dict]]
    df.columns = [mapping_dict[col.split(":")[0]] for col in df.columns]
    sample_name=sample_name.split('_')[0]
    df['sample']=sample_name
    df['cell_id']=df.index
    df.reset_index(drop=True,inplace=True)
    print(df.head())
    if i==0:
        data=pd.concat([data,df])
        i=i+1
    else:
        data=pd.concat([data,df],axis=0,join='inner')

print("Done!")

Processing CRC0322_NT_1_3000...
     TSPAN6  TNMD      DPM1  SCYL3  C1orf112     FUCA2      GCLC      NFYA  \
0  7.360118   0.0  8.355721    0.0  0.000000  8.355721  7.360118  0.000000   
1  7.679176   0.0  8.675652    0.0  0.000000  7.679176  9.259438  0.000000   
2  7.046006   0.0  0.000000    0.0  0.000000  0.000000  8.040537  0.000000   
3  8.466266   0.0  6.889439    0.0  5.901557  5.901557  7.883342  0.000000   
4  8.332733   0.0  7.750006    0.0  0.000000  6.756692  6.756692  6.756692   

   STPG1    NIPAL3  ...  BLACAT1  AC006978.2  AL512506.3  AC106886.6  \
0    0.0  0.000000  ...      0.0         0.0         0.0         0.0   
1    0.0  0.000000  ...      0.0         0.0         0.0         0.0   
2    0.0  0.000000  ...      0.0         0.0         0.0         0.0   
3    0.0  5.901557  ...      0.0         0.0         0.0         0.0   
4    0.0  0.000000  ...      0.0         0.0         0.0         0.0   

   AL157392.5  AC004706.3  AL031777.2  LBHD2   sample             

In [31]:
geni_senza_na = data.columns[data.isnull().sum() == 0]
geni_con_na=data.columns[data.isnull().sum() > 0]
print(f"Geni senza NaN: {len(geni_senza_na)}")
print(f"Geni con NaN: {len(geni_con_na)}")

Geni senza NaN: 14197
Geni con NaN: 0


In [64]:
gene_expr = data.drop(columns=["sample", "cell_id"])
cell_uid = data["sample"] + "_" + data["cell_id"]

adata = ad.AnnData(X=gene_expr.values)
adata.var_names = gene_expr.columns

# Aggiungi metadati
adata.obs["sample"] = data["sample"].values
adata.obs["cell_id"] = data["cell_id"].values
adata.obs_names = cell_uid.values  

# Calcolo geni variabili
sc.pp.highly_variable_genes(adata, n_top_genes=4000, flavor='cell_ranger')
adata = adata[:, adata.var["highly_variable"]]

In [65]:
geni_variabili = adata.var_names[adata.var['highly_variable']].tolist()
grafo_path='Graphs/grafo_filtrato_undirect_small.csv'

grafo=pd.read_csv(grafo_path)
geni=grafo['source'].unique()

In [66]:
geni_comuni=[gene for gene in geni_variabili if gene in geni]

In [67]:
len(geni_comuni)

354

In [56]:
deg_path='/mnt/trcanmed/snaketree/stash/degK.tsv'

#egrassi@godot:/scratch/trcanmed/DE_RNASeq/dataset/scRNA_deg$ cat KRAS_cutoff0.05-KRAS.vs.WT.goinsplit_down.tsv KRAS_cutoff0.05-KRAS.vs.WT.goinsplit_up.tsv > /mnt/trcanmed/snaketree/stash/degK.tsv

deg_=pd.read_csv(deg_path)
geni_deg=deg_['gene'].unique()

In [57]:
geni_comuni_deg=[gene for gene in geni_comuni if gene in geni_deg]

In [58]:
len(geni_deg)

620

In [70]:
geni_comuni_deg=[gene for gene in geni_variabili if gene in geni_deg]
len(geni_comuni_deg)

174